# TAM Research — isolated RLT Colab bridge

This notebook runs only the `exp/rlt-colab` experiment lane. It polls bounded JSON jobs from GitHub and writes create-once terminal result JSON files back to that same branch. It does not touch `main`, historical result directories, or existing Modal workflows.

**Before running:** Runtime → Change runtime type → choose a GPU. In the Colab Secrets panel add `GITHUB_TOKEN`, scoped only to `vinceackermann2-sys/tam-research` with repository **Contents: Read and write** permission.


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before starting the bridge.')
print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Missing Colab secret GITHUB_TOKEN')

repo_dir = Path('/content/tam-rlt-bridge')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

askpass = Path('/content/rlt-git-askpass.sh')
askpass.write_text('''#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n''')
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

env = os.environ.copy()
env['GITHUB_TOKEN'] = token
env['GIT_ASKPASS'] = str(askpass)
env['GIT_ASKPASS_REQUIRE'] = 'force'
env['GIT_TERMINAL_PROMPT'] = '0'

subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch',
    '--branch', 'exp/rlt-colab',
    'https://github.com/vinceackermann2-sys/tam-research.git',
    str(repo_dir),
], check=True, env=env)
askpass.unlink(missing_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True, env=env)
print('bridge code pinned to:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip())


Optional: if you want datasets/checkpoints to survive a Colab disconnect, mount Drive and set `RLT_DATA_DIR` / `RLT_RUN_ROOT` before starting the worker. By default they stay in `/content/rlt-data` and `/content/rlt-runs` and never enter the repository.


In [ ]:
# Optional persistent storage example — leave commented unless you want it.
# from google.colab import drive
# drive.mount('/content/drive')
# env['RLT_DATA_DIR'] = '/content/drive/MyDrive/tam-rlt/data'
# env['RLT_RUN_ROOT'] = '/content/drive/MyDrive/tam-rlt/runs'


In [ ]:
# Keep this cell running. Stop it to disconnect the bridge.
worker = repo_dir / 'experiments' / 'rlt' / 'bridge_worker.py'
subprocess.run([
    sys.executable, str(worker),
    '--branch', 'exp/rlt-colab',
    '--poll-seconds', '20',
], check=True, cwd=str(repo_dir), env=env)
